In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/nigeria_multisensor_dataset.csv')

# Geopolitical zone mapping for spatial CV
GEO_ZONES = {
    'North Central': ['Benue', 'Kogi', 'Kwara', 'Nasarawa', 'Niger', 'Plateau', 'Abuja', 'Federal Capital Territory'],
    'North East': ['Adamawa', 'Bauchi', 'Borno', 'Gombe', 'Taraba', 'Yobe'],
    'North West': ['Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Sokoto', 'Zamfara'],
    'South East': ['Abia', 'Anambra', 'Ebonyi', 'Enugu', 'Imo'],
    'South South': ['Akwa Ibom', 'Bayelsa', 'Cross River', 'Delta', 'Edo', 'Rivers'],
    'South West': ['Ekiti', 'Lagos', 'Ogun', 'Ondo', 'Osun', 'Oyo']
}

zone_map = {state: zone for zone, states in GEO_ZONES.items() for state in states}
df['geo_zone'] = df['state_name'].map(zone_map)
df['geo_zone_id'] = pd.factorize(df['geo_zone'])[0]

df['log_viirs'] = np.log1p(df['viirs_mean_radiance'])

# Define feature spaces
X_2features = df[['log_viirs', 'modis_mean_ndvi']].values
X_all = df[['log_viirs', 'modis_mean_ndvi', 's2_mean_ndbi', 's2_mean_mndwi', 's2_mean_bsi']].values
y = df['poverty_rate_pct'].values
groups = df['geo_zone_id'].values

print(f"Loaded {len(df)} states across 6 geopolitical zones.")

Loaded 37 states across 6 geopolitical zones.


In [2]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

logo = LeaveOneGroupOut()

models = {
    'OLS (2-features: VIIRS + NDVI)': (LinearRegression(), X_2features),
    'OLS (Full Multi-Sensor: +NDBI, MNDWI, BSI)': (LinearRegression(), X_all),
    'Ridge (Full Multi-Sensor)': (Ridge(alpha=2.0), X_all),
    'Random Forest (Full Multi-Sensor)': (RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42), X_all)
}

comparison = []
for name, (model, X_mat) in models.items():
    preds = cross_val_predict(model, X_mat, y, cv=logo, groups=groups)
    r2 = r2_score(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    mae = mean_absolute_error(y, preds)
    
    comparison.append({
        'Model Configuration': name,
        'Spatial CV R²': round(r2, 3),
        'Spatial CV RMSE (% pts)': round(rmse, 2),
        'Spatial CV MAE (% pts)': round(mae, 2)
    })

df_res = pd.DataFrame(comparison)
print("=== Spatial Leave-One-Zone-Out Cross-Validation Benchmark ===")
print(df_res.to_string(index=False))

=== Spatial Leave-One-Zone-Out Cross-Validation Benchmark ===
                       Model Configuration  Spatial CV R²  Spatial CV RMSE (% pts)  Spatial CV MAE (% pts)
            OLS (2-features: VIIRS + NDVI)          0.810                    10.56                    8.43
OLS (Full Multi-Sensor: +NDBI, MNDWI, BSI)          0.805                    10.71                    8.60
                 Ridge (Full Multi-Sensor)          0.261                    20.83                   17.75
         Random Forest (Full Multi-Sensor)          0.436                    18.20                   13.82


In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

ols_std = LinearRegression()
ols_std.fit(X_scaled, y)

feature_names = ['log(VIIRS)', 'MODIS NDVI', 'S2 NDBI (Built-up)', 'S2 MNDWI (Water)', 'S2 BSI (Bare Soil)']
importance = pd.DataFrame({
    'Feature': feature_names,
    'Standardized Coef': ols_std.coef_
}).sort_values('Standardized Coef', key=abs, ascending=False)

print("\nStandardized Effect Size on Poverty Headcount (%):")
print(importance.to_string(index=False))


Standardized Effect Size on Poverty Headcount (%):
           Feature  Standardized Coef
S2 BSI (Bare Soil)          83.834634
S2 NDBI (Built-up)         -75.202409
        log(VIIRS)         -13.878103
        MODIS NDVI          -4.237624
  S2 MNDWI (Water)           3.737454
